# Schema Analysis Report Generator
This notebook uses the Vanna Agent to autonomously analyze the `Chinook.sqlite` database and generate a comprehensive schema report.

In [1]:
import os
from vanna import Agent, AgentConfig
from vanna.integrations.openai import OpenAILlmService
from vanna.integrations.sqlite import SqliteRunner
from vanna.tools import RunSqlTool
from vanna.core.registry import ToolRegistry
from vanna.core.user import UserResolver, User, RequestContext
from vanna.integrations.local.agent_memory import DemoAgentMemory

In [2]:
# Configuration
DB_PATH = "./Chinook.sqlite"
# Ensure the database exists (download if needed as per quickstart)
if not os.path.exists(DB_PATH):
    import httpx
    print(f"Downloading {DB_PATH}...")
    with open(DB_PATH, "wb") as f:
        with httpx.stream("GET", "https://vanna.ai/Chinook.sqlite") as response:
            for chunk in response.iter_bytes():
                f.write(chunk)
    print("Download complete.")

In [3]:
# Initialize Components

# 1. User Resolver (Simple mock for local use)
class LocalUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        return User(id="local_admin", email="admin@localhost", group_memberships=['admin'])

# 2. LLM Service (LM Studio)
llm = OpenAILlmService(
    model="qwen/qwen3-coder-30b", # Adjust model name as needed
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio"
)

# 3. Tools
tools = ToolRegistry()
sql_runner = SqliteRunner(database_path=DB_PATH)
tools.register_local_tool(RunSqlTool(sql_runner=sql_runner), access_groups=['admin'])

# 4. Memory
agent_memory = DemoAgentMemory()

# 5. Agent
agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=LocalUserResolver(),
    config=AgentConfig(max_tool_iterations=100),
    agent_memory=agent_memory
)

In [ ]:
# Run Analysis
import asyncio

async def generate_report():
    dataset_analysis_prompt = """
You are a Senior Data Engineer. Your task is to analyze the database schema and generate a comprehensive markdown report.

Follow these steps:
1.  Inspect the database to list all tables (hint: query `sqlite_master` or `information_schema`).
2.  For each table, inspect the columns, data types, and any primary/foreign keys.
3.  Infer relationships between tables based on naming conventions (e.g., `ArtistId` in `Albums` table).
4.  Generate a report in Markdown format with the following structure:
    *   **Executive Summary**: Brief overview of the database domain (e.g., Music Store).
    *   **Entity-Relationship Diagram**: Use Mermaid JS syntax to visualize relationships.
    *   **Table Details**: A section for each table with a table of columns and descriptions.
    *   **Data Integrity suggestions**: Any potential missing keys or oddities.

Output the FINAL REPORT as a single markdown block and in Chinese.
"""

    print("Starting analysis... (this may take a minute)")
    
    # We use a dummy request context for the local user
    request_context = RequestContext()
    
    final_response = ""
    
    # Iterate through the agent's response stream
    async for component in agent.send_message(request_context, dataset_analysis_prompt):
        if hasattr(component, 'rich_component') and component.rich_component:
            # Capture the final content (assuming the LLM generates it in rich text or simple text)
            # For complex flows, we might need to aggregate text.
            if hasattr(component.rich_component, 'content'):
                print(component.rich_component.content, end="", flush=True)
                final_response += str(component.rich_component.content)
        elif hasattr(component, 'simple_component') and component.simple_component:
             print(component.simple_component.text, end="\n", flush=True)

    # Save report
    with open("schema_report.md", "w") as f:
        f.write(final_response)
    print("\n\nAnalysis complete. Report saved to schema_report.md")

# Run the async function in the notebook loop
await generate_report()

Starting analysis... (this may take a minute)
I'll analyze the database schema to generate a comprehensive report. Let me start by inspecting the database structure.

First, I'll list all tables in the database:
Now I'll examine the schema of each table to understand their columns, data types, and relationships.

Let me get more detailed information about the Album table structure:

Let me check the Artist table structure:

Let me check the Customer table structure:

Let me check the Employee table structure:

Let me check the Genre table structure:

Let me check the Invoice table structure:

Let me check the InvoiceLine table structure:

Let me check the MediaType table structure:

Let me check the Playlist table structure:

Let me check the PlaylistTrack table structure:

Let me check the Track table structure:

# Database Schema Analysis Report

## Executive Summary

This database appears to be a music store management system that manages artists, albums, tracks, customers, employee